In [ ]:
import pandas as pd
import scipy.stats as stats
import altair as alt
import numpy as np
alt.data_transformers.disable_max_rows()

# Data Processing Helper Functions

In [ ]:
def spliceai_mapper(df):

    df['maxSpliceAI'] = df[['spliceAI_DS_AG', 'spliceAI_DS_AL', 'spliceAI_DS_DG', 'spliceAI_DS_DL']].max(axis = 1)

    df['splice_impact'] = '< Threshold'
    SPLICE_IMPACT_COLS = {
        'spliceAI_DS_AG': 'Acceptor Gain',
        'spliceAI_DS_AL': 'Acceptor Loss',
        'spliceAI_DS_DG': 'Donor Gain',
        'spliceAI_DS_DL': 'Donor Loss',
    }

    for col, label in SPLICE_IMPACT_COLS.items():
        df.loc[df[col] >= 0.2, 'splice_impact'] = label
    
    return df

In [ ]:
def molecular_consequence_mapper(df, remap_col):

    df = df.dropna(subset=[remap_col]).copy()
    CONSEQUENCE_EXACT = {
            'synonymous_variant': 'Synonymous',
            'intron_variant':     'Intron',
            'stop_gained':        'Stop Gained',
            'stop_lost':          'Stop Lost',
            'start_lost':         'Start Lost',
            'inframe_indel':      'Inframe Indel',
        }

    CONSEQUENCE_CONTAINS = {
        'missense': 'Missense',
        'site':     'Canonical Splice',
        'ing_var':  'Splice Region',
        'UTR':      'UTR Variant',
    }

    df[remap_col] = df[remap_col].replace(CONSEQUENCE_EXACT)
    for pattern, label in CONSEQUENCE_CONTAINS.items():
        df.loc[df[remap_col].str.contains(pattern), remap_col] = label
    

    return df

In [ ]:
raw_df = pd.read_excel('/Users/ivan/Documents/GitHub/PillarProject_CAVA_Analysis/Data/sge_data_for_qc/raw_scores_for_rna/external_rna_data/20260424_CAVASGE_SangerSGE_FindlaySGE.xlsx')

raw_df['auth_reported_func_class'] = raw_df['auth_reported_func_class'].replace({
    'LOF':           'functionally_abnormal',
    'LOF1':          'functionally_abnormal',
    'LOF2':          'functionally_abnormal',
    'depleted':      'functionally_abnormal',
    'slow depleted': 'functionally_abnormal',
    'fast depleted': 'functionally_abnormal',
    'slow depleting':'functionally_abnormal',
    'fast depleting':'functionally_abnormal',
    'enriched':      'functionally_normal',
    'FUNC':          'functionally_normal',
    'Neutral':       'functionally_normal',
    'unchanged':     'functionally_normal',
    'INT':           'indeterminate',
    'Intermediate':  'indeterminate',
})

df = spliceai_mapper(raw_df)
df = molecular_consequence_mapper(df, 'simplified_consequence')

df.head()

# Intial Data Processing and Z-Score Normalization

In [ ]:
normal_mask = df['auth_reported_func_class'] == 'functionally_normal'

df['z_score'] = df.groupby('Gene')['auth_reported_score'].transform(
    lambda x: (x - x[normal_mask.reindex(x.index)].mean()) 
              / x[normal_mask.reindex(x.index)].std()
)

df = df[['Gene', 'auth_reported_score', 'auth_reported_func_class', 'z_score', 
         'simplified_consequence', 'maxSpliceAI', 'splice_impact']].dropna(subset=['auth_reported_func_class'])

df = df.dropna(subset=['auth_reported_func_class'])
print(pd.unique(df['Gene']))

## Z-score Normalization Sanity Check

In [ ]:
z_score_plot = alt.Chart(df).mark_boxplot().encode(
    x='Gene',
    y='z_score:Q'
).facet('auth_reported_func_class')

z_score_plot.display()

# Intron Variants, Z-score normalized

In [ ]:
# Scatter plot helper function

def corr_scatter(df, consequence, rep1, rep2, gene):

    rep1_max = df[rep1].max(axis = 0)
    rep2_max = df[rep2].max(axis = 0)
    rep2_min = df[rep2].min(axis = 0)

    x_max = rep1_max * 1.05
    y_max = rep2_max * 1.05
    y_min = rep2_min * 1.05

    df = df.dropna(subset = [rep1, rep2]).copy()
    df = df.loc[df['simplified_consequence']==consequence]
    
    scatter = alt.Chart(df).mark_circle().encode(
        x = alt.X(f'{rep1}:Q',
                  scale = alt.Scale(0, x_max)
                  ),
        y = alt.Y(f'{rep2}:Q',
                  scale = alt.Scale(y_min, y_max)
                  ),
        color='splice_impact:N',
        tooltip = ['Gene', 'auth_reported_score']
    )

    corr,_=stats.pearsonr(df[rep1], df[rep2])

    r_text = alt.Chart(pd.DataFrame({
        rep1: [x_max * 0.95],
        rep2: [1],
        'text': [f'r = {corr:.3f}']
    })).mark_text(
        align='right',
        baseline='bottom',
        fontSize=18,
        fontWeight='bold',
        color='black'
    ).encode(
        x = alt.X(f'{rep1}:Q',
                  scale = alt.Scale(0, x_max)),
        y = alt.Y(f'{rep2}:Q',
                  scale = alt.Scale(y_min, y_max)
                  ),
        text='text:N'
    )

    scatter = (scatter+r_text).properties(title = gene).resolve_scale(x = 'shared', y = 'shared').display()

    rep_test = f'{rep1} vs. {rep2}'
    return scatter, gene, rep_test, corr

In [ ]:
scatter, _, _, _ = corr_scatter(df, 'Intron', 'z_score', 'maxSpliceAI', 'All SGE Intron Variants')

In [ ]:
scatter, _, _, _ = corr_scatter(df, 'Missense', 'z_score', 'maxSpliceAI', 'All SGE Intron Variants')